# Regions of Interest

In neuroim there is comprehensive support for creating and working with regions of interest (ROI).

## Creating a spherical ROI

To create a spherical ROI around a central point, we need an existing `NeuroVol` or `NeuroSpace` object.

### Basic spherical ROI

To create a spherical region of interest with a 5mm radius around a central voxel at (20, 20, 20)

In [ ]:
# Import required libraries
import neuroim as pn
import numpy as np

# Create example data for tutorial
# (In practice, you would use real neuroimaging files)

# Create example volume
space_3d = pn.NeuroSpace(dim=(64, 64, 25), spacing=(3.5, 3.5, 5.0), 
                         origin=(-110.5, -88.9342, -42.75))
example_data = np.random.randn(64, 64, 25)
example_data = (example_data - example_data.min()) / (example_data.max() - example_data.min())

# Create volume
vol = pn.DenseNeuroVol(example_data, space_3d)

# In practice, you would read a NIFTI file:
# vol = pn.read_vol("brain_image.nii.gz")

# Create a spherical ROI centered at voxel [20, 20, 10] with 5mm radius
sphere = pn.spherical_roi(vol, [20, 20, 10], radius=5, fill=100)
print(sphere)
print(f"ROI contains {len(sphere)} voxels")

The `fill` parameter sets all values in the ROI to 100. If omitted, the ROI will contain the original values from the volume.

In [ ]:
# Real-world coordinate in mm
rpoint = np.array([-34, -28, 10])

# Convert to voxel coordinates
vox = vol.space.coord_to_grid(rpoint.reshape(1, -1))[0]

# Create spherical ROI with 10mm radius
sphere = pn.spherical_roi(vol, vox, radius=10, fill=1)
print(f"ROI contains {len(sphere)} voxels")

Now we can verify the center by converting back to real-world coordinates

In [ ]:
# Get all coordinates in the ROI
roi_world_coords = sphere.get_coords(real=True)

# Calculate center of mass
center_of_mass = np.mean(roi_world_coords, axis=0)
print(f"Center of mass: {center_of_mass}")

## Converting an ROI to a SparseNeuroVol

In [ ]:
# Create a spherical ROI
sphere = pn.spherical_roi(vol, [32, 32, 12], radius=10, fill=1)
print(sphere)

# Convert to SparseNeuroVol
sparsevol = sphere.as_sparse()
print(f"Sum of ROI values: {np.sum(sparsevol.data)}")
print(f"Sum matches: {np.sum(sparsevol.data) == np.sum(sphere.data)}")
print(f"Dimensions match: {sparsevol.shape == vol.shape}")

## Creating other ROI shapes

In [ ]:
# Create a 7x7 square ROI in the z=10 plane
square = pn.square_roi(vol, centroid=[30, 30, 10], 
                       surround=3, fixdim=2, fill=1)
print(f"Square ROI has {len(square)} voxels")

## Cuboid ROIs

In [ ]:
# Create a cuboid with equal dimensions
cube = pn.cuboid_roi(vol, centroid=[30, 30, 12], 
                     surround=5, fill=1)

# Create an asymmetric cuboid
cuboid = pn.cuboid_roi(vol, centroid=[30, 30, 12], 
                       surround=[3, 4, 5], fill=1)
print(f"Cuboid ROI has {len(cuboid)} voxels")

## Working with multiple ROIs

In [ ]:
# Define multiple ROI centers
centers = np.array([[20, 20, 10],
                    [40, 40, 15],
                    [30, 50, 12]])

# Create ROIs with different values
roi_list = pn.spherical_roi_set(vol, centers, radius=6, 
                                 fill=[100, 200, 300])
print(f"Created {len(roi_list)} ROIs")
for i, roi in enumerate(roi_list):
    print(f"ROI {i}: {len(roi)} voxels, value={roi.data[0]}")

## Simple searchlight analysis

In [ ]:
# Generate searchlight ROIs around each voxel
from neuroim import searchlight

# Create a brain mask
mask = vol.data > 0.2
mask_vol = pn.LogicalNeuroVol(mask, vol.space)

# Create searchlights with 8mm radius
slist = searchlight(mask_vol, radius=8)

# Compute mean value in each searchlight (first 100 for demo)
means = []
for roi in list(slist)[:100]:
    # Get values from volume at ROI coordinates
    values = vol[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]]
    means.append(np.mean(values))

print(f"Computed means for {len(means)} searchlights")

## Random searchlight

In [ ]:
from neuroim import random_searchlight

# Create random searchlights
rois = random_searchlight(mask_vol, n=50, radius=8)

# Process each ROI
results = []
for roi in rois:
    values = vol[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]]
    results.append(np.mean(values))

print(f"Processed {len(results)} random searchlights")

## Clustered searchlight

In [ ]:
# First create a clustering
mask_indices = np.where(vol.data > 0.2)
coords = np.column_stack(mask_indices)

# Simple k-means clustering
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=20, random_state=42)
labels = kmeans.fit_predict(coords)

# Create ClusteredNeuroVol
cluster_data = np.zeros_like(vol.data)
cluster_data[mask_indices] = labels + 1  # Labels start at 1
kvol = pn.ClusteredNeuroVol(cluster_data, vol.space)

# Create ROIs from clusters
from neuroim import clustered_searchlight
cluster_rois = clustered_searchlight(kvol)

# Analyze each cluster
cluster_means = []
for roi in cluster_rois:
    values = vol[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]]
    cluster_means.append(np.mean(values))

print(f"Analyzed {len(cluster_means)} clusters")

## ROI-based time series extraction

In [ ]:
# Create 4D data for demo
space_4d = pn.NeuroSpace(dim=(64, 64, 25, 100))
vec_data = np.random.randn(64, 64, 25, 100)
vec = pn.DenseNeuroVec(vec_data, space_4d)

# In practice:
# vec = pn.read_vec("functional_data.nii")

# Create ROI
roi = pn.spherical_roi(vec.space, [32, 32, 12], radius=10)

# Extract time series for all voxels in ROI
roi_timeseries = vec.series_roi(roi)
print(f"Time series shape: {roi_timeseries.shape}")

# Compute mean time series
mean_ts = np.mean(roi_timeseries, axis=1)

# Plot
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(mean_ts)
plt.xlabel('Time point')
plt.ylabel('Mean signal')
plt.title('Mean ROI time series')
plt.show()

## Saving and loading ROIs

In [ ]:
# Create ROI
roi = pn.spherical_roi(vol, [30, 30, 12], radius=8)

# Convert to sparse volume for saving
sparse_roi = roi.as_sparse()

# In practice, you would save it:
# pn.write_vol(sparse_roi, "my_roi.nii.gz")

# For demo, let's show how to reconstruct
# Get ROI coordinates and values
roi_indices = sparse_roi.indices
roi_values = sparse_roi.data

# Convert indices to coordinates
roi_coords = vol.space.index_to_grid(roi_indices)

# Reconstruct ROI
reconstructed_roi = pn.ROIVol(roi_values, vol.space, roi_coords)
print(f"Original ROI: {len(roi)} voxels")
print(f"Reconstructed ROI: {len(reconstructed_roi)} voxels")